# Bicycle and BB codes: dimensions, direct sums, and boundary choices

**Question.** How do polynomial inputs determine K, why are two length-144 examples different, and what do basis permutations actually certify?

**Paper locations.** Section IX, Eq. (90) for the quotient dimension and Eqs. (93)–(94) for the BB144 polynomials (`sec:BB`, `eq:bb-logical-qubit-quotient`, `eq:fg`).

**Companion paper.** Yuan Yao, Guo Zhang, Yingli Yang, Yuanye Zhu, Xiaosi Xu, Xiao Yuan, and Ying Li, **“An introduction to qLDPC codes from a structural perspective.”** Submitted.

The paper numbers refer to the manuscript identified by `PAPER_SOURCE` in
[`paper_cases.py`](../qldpc_generator/paper_cases.py). Source labels are retained
to help locate the same material after renumbering. Equation numbers below are
local to this notebook; code outputs are computational results.

**Before running.** Follow the [installation instructions](../README.md#installation-and-execution),
select that Python environment, and run the cells in order. All matrix arithmetic
and ranks are over GF(2); matrix indices start at zero. For the coverage overview,
start with [06 — Paper validation](06-paper-validation.ipynb).

In [1]:
import numpy as np
from IPython.display import Markdown, display

def show_table(headers, rows):
    """Format results for reading; no mathematical calculations happen here."""
    lines = ["| " + " | ".join(map(str, headers)) + " |",
             "| " + " | ".join(["---"] * len(headers)) + " |"]
    lines += ["| " + " | ".join(map(str, row)) + " |" for row in rows]
    display(Markdown("\n".join(lines)))

## 1. Start with the common bicycle check form

For square binary blocks A and B of the same size that commute, the bicycle
form gives a CSS code:

$$
H_X=(A\;B),\qquad H_Z=(B^{\mathsf T}\;A^{\mathsf T}),\qquad
H_XH_Z^{\mathsf T}=AB+BA=0. \tag{1}
$$

Standard bicycle codes set B = Aᵀ. Generalized bicycle (GB) codes allow two
cyclic polynomials, whose circulant matrices commute. Here are all inputs for
the two small examples included in the validation report:

$$
\begin{aligned}
\text{bicycle6: }&a=1+x,\quad L=3,\quad B=A^{\mathsf T},\\
\text{GB48: }&a=1+x^2+x^8+x^{15},\quad
b=1+x^2+x^{12}+x^{17},\quad L=24.
\end{aligned} \tag{2}
$$

For GB, a polynomial gcd provides a dimension check independent of binary
matrix ranks:

$$
K=2\deg\gcd(a,b,x^L-1). \tag{3}
$$

In [2]:
import sympy as sp
from qldpc_generator.algebra import x, y
from qldpc_generator.constructions import (
    standard_bicycle, generalized_bicycle, gb_dimension, bivariate_bicycle,
    bb_quotient_dimension, generalized_toric, coprime_bicycle, toric, planar_surface,
)
from qldpc_generator.paper_cases import bb144_case, bb_family, bb_direct_sum_equivalence
from qldpc_generator.equivalences import monomial_bb_equivalence, coprime_gb_bb_equivalence

small = standard_bicycle(1+x, 3)
a = 1+x**2+x**8+x**15
b = 1+x**2+x**12+x**17
gb48 = generalized_bicycle(a, b, 24)
assert gb48.k == gb_dimension(a, b, 24) == 6
rows = []
for name, instance, expected in [('bicycle6', small, (6, 2, 2)), ('GB48', gb48, (48, 6, 8))]:
    distance = instance.distance(expected[2])
    assert distance['status'] == 'exact'
    assert (instance.n, instance.k, distance['value']) == expected
    rows.append([name, instance.n, instance.k, f"D = {distance['value']}", distance['status']])
show_table(["Instance", "N", "K", "Local distance", "Status"], rows)

| Instance | N | K | Local distance | Status |
| --- | --- | --- | --- | --- |
| bicycle6 | 6 | 2 | D = 2 | exact |
| GB48 | 48 | 6 | D = 8 | exact |

## 2. BB144: specify the ring, not just the polynomials

BB uses two commuting cyclic coordinates. The periods are part of the code
definition. Paper Eq. (90) expresses K as a quotient dimension:

$$
R_{L,M}=\frac{\mathbb F_2[x^{\pm1},y^{\pm1}]}{(x^L-1,y^M-1)},
\qquad K=2\dim_{\mathbb F_2}R_{L,M}/(f,g). \tag{4}
$$

Because x and y become invertible under the period relations, the dimension
can also be computed in the ordinary polynomial ring after imposing those
relations. Over GF(2), minus and plus coincide. For the paper's BB144 instance,

$$
(L,M)=(12,6),\qquad f=y^2(x+x^2+y^3),\qquad
g=x^2(y+y^2+x^3). \tag{5}
$$

We compare two calculations: binary ranks of the check matrices and the
number of standard monomials of a Groebner basis. A standard monomial is
one not divisible by any leading monomial of that basis; its residue class
contributes one basis vector of the quotient.

In [3]:
L, M = 12, 6
f = y**2*(x+x**2+y**3)
g = x**2*(y+y**2+x**3)
reference = bivariate_bicycle(f, g, L, M)
stored = bb144_case()
assert np.array_equal(reference.hx, stored.hx) and np.array_equal(reference.hz, stored.hz)
quotient_dimension = bb_quotient_dimension(f, g, L, M)

# Expose the symbolic basis underlying the quotient calculation for this input.
basis = sp.groebner([f, g, x**L+1, y**M+1], x, y, modulus=2)
leading = [polynomial.monoms()[0] for polynomial in basis.polys]
standard = [x**i*y**j for i in range(L) for j in range(M)
            if not any(i >= a and j >= b for a, b in leading)]
assert len(standard) == quotient_dimension == 6
p = reference.parameters()
assert (reference.n, reference.k, p['rank_x'], p['rank_z']) == (144, 12, 66, 66)
assert reference.k == 2*quotient_dimension
print("Groebner basis:", [polynomial.as_expr() for polynomial in basis.polys])
print("Standard monomials:", standard)
show_table(["Independent calculation", "Result for K"], [
    ["144 - rank_2(H_X) - rank_2(H_Z)", reference.k],
    ["2 * quotient dimension", 2*quotient_dimension],
])
reference_distance = reference.distance(6)
assert reference_distance['status'] == 'bounded' and reference_distance['lower_bound'] == 7
print("BB144 distance: D >= 7 locally; D = 12 is the literature value.")

Groebner basis: [x**2 + x + y**3, x*y**2 + x*y + x + y**3 + 1, y**4 + y**2 + 1]
Standard monomials: [1, y, y**2, y**3, x, x*y]


| Independent calculation | Result for K |
| --- | --- |
| 144 - rank_2(H_X) - rank_2(H_Z) | 12 |
| 2 * quotient dimension | 12 |

BB144 distance: D >= 7 locally; D = 12 is the literature value.


## 3. A different BB family: direct sums of BB72

The next family is not an alternative description of the BB144 input above.
It uses different polynomials, although its t = 2 member also has 144 qubits:

$$
f_t=x^{3t}+y+y^2,\qquad g_t=y^3+x^t+x^{2t},\qquad
(L,M)=(6t,6). \tag{6}
$$

Every horizontal shift preserves i modulo t, including the inverse shifts
in Z checks. Writing i = r+tu separates the lattice into t components indexed
by r. After relabelling checks and qubits, each component is the t = 1 code.
The library returns the complete basis permutation, not only matching ranks.

The base block has exact distance six. A direct sum has additive K and the
minimum of its block distances, so this decomposition gives

$$
\mathcal C_t\cong\bigoplus_{r=0}^{t-1}\mathcal C_1,\qquad
[[N,K,D]]=[[72t,12t,6]]. \tag{7}
$$

The argument about residues explains the family decomposition. The following
matrix checks verify its implementation at t = 1, 2, 3, and 4; they are not a
replacement for that general argument.

In [4]:
base_distance = bb_family(1).distance(6)
assert base_distance['status'] == 'exact' and base_distance['value'] == 6
rows = []
for t in (1, 2, 3, 4):
    eq = bb_direct_sum_equivalence(t)
    assert eq.verify() and (eq.source.n, eq.source.k) == (72*t, 12*t)
    # Check explicitly that every target row stays inside its own 72-qubit block.
    for checks in (eq.target.hx, eq.target.hz):
        for i, row in enumerate(checks):
            assert set(np.flatnonzero(row)//72) == {i//36}
    rows.append([t, str(x**(3*t)+y+y**2), str(y**3+x**t+x**(2*t)),
                 eq.source.n, eq.source.k, "D = 6 (direct sum)", eq.verify()])
show_table(["t", "f_t", "g_t", "N", "K", "Distance", "Both matrix identities"], rows)
show_table(["Horizontal coordinate i (t=2)", "Component r", "Coordinate u within component"], [
    [i, i % 2, i//2] for i in range(12)
])

| t | f_t | g_t | N | K | Distance | Both matrix identities |
| --- | --- | --- | --- | --- | --- | --- |
| 1 | x**3 + y**2 + y | x**2 + x + y**3 | 72 | 12 | D = 6 (direct sum) | True |
| 2 | x**6 + y**2 + y | x**4 + x**2 + y**3 | 144 | 24 | D = 6 (direct sum) | True |
| 3 | x**9 + y**2 + y | x**6 + x**3 + y**3 | 216 | 36 | D = 6 (direct sum) | True |
| 4 | x**12 + y**2 + y | x**8 + x**4 + y**3 | 288 | 48 | D = 6 (direct sum) | True |

| Horizontal coordinate i (t=2) | Component r | Coordinate u within component |
| --- | --- | --- |
| 0 | 0 | 0 |
| 1 | 1 | 0 |
| 2 | 0 | 1 |
| 3 | 1 | 1 |
| 4 | 0 | 2 |
| 5 | 1 | 2 |
| 6 | 0 | 3 |
| 7 | 1 | 3 |
| 8 | 0 | 4 |
| 9 | 1 | 4 |
| 10 | 0 | 5 |
| 11 | 1 | 5 |

The two length-144 rows below must not be conflated. Their different K values
are expected because they are different inputs. This direct-sum example shows
that BB structure alone does not force K to remain constant as N grows; its
growing K comes with disconnected fixed-size blocks and constant distance.

In [5]:
show_table(["Length-144 instance", "Polynomial specification", "K", "Distance evidence"], [
    ["Paper BB144", "Eq. (5) above", reference.k, "D >= 7 locally; literature D = 12"],
    ["Direct-sum family at t=2", "Eq. (6) above", bb_family(2).k, "D = 6 from two BB72 blocks"],
])

| Length-144 instance | Polynomial specification | K | Distance evidence |
| --- | --- | --- | --- |
| Paper BB144 | Eq. (5) above | 12 | D >= 7 locally; literature D = 12 |
| Direct-sum family at t=2 | Eq. (6) above | 24 | D = 6 from two BB72 blocks |

## 4. Optional: translations and coprime coordinates

Two further correspondences in Section IX reuse the same check form.
First, multiplying each template by a monomial translates its support.
On the 12-by-6 torus, the translated BB144 polynomials are

$$
f'=x^{-1}y^{-2}f=1+x+x^{-1}y^{-3},\qquad
g'=x^{-2}y^{-1}g=1+y+x^3y^{-1}. \tag{8}
$$

The equalities use the period relations. This gives the generalized toric
description with exponents (-1,-3,3,-1). It is an actual correspondence to
the reference BB144, unlike the direct-sum example.

Second, coprime periods allow a single cyclic coordinate to enumerate the
whole torus. For periods 3 and 5 the Chinese remainder map is

$$
\mathbb Z_{15}\longrightarrow\mathbb Z_3\times\mathbb Z_5,
\qquad k\longmapsto(k\bmod3,k\bmod5),\qquad x_{\rm cyclic}\longmapsto xy. \tag{9}
$$

In [6]:
translated = generalized_toric(-1, -3, 3, -1, 12, 6)
translation = monomial_bb_equivalence(reference, translated, 12, 6, (-1, -2), (-2, -1))
assert translation.verify()
cyclic = generalized_bicycle(1+x+x**5, 1+x**2, 15)
coprime = coprime_bicycle(1+x+x**5, 1+x**2, 3, 5)
crt = coprime_gb_bb_equivalence(cyclic, coprime, 3, 5)
assert crt.verify()
assert len({(k % 3, k % 5) for k in range(15)}) == 15
show_table(["Correspondence", "N", "K", "Both matrix identities with shared P1"], [
    ["BB144 -> generalized toric", reference.n, reference.k, translation.verify()],
    ["GB period 15 -> BB periods (3,5)", cyclic.n, cyclic.k, crt.verify()],
])

# Failure control: noncoprime periods cannot use this single-cycle identification.
try:
    coprime_bicycle(1+x, 1+x, 4, 6)
except ValueError as exc:
    assert 'Coprime' in str(exc)
    print("Expected refusal for periods (4,6):", exc)
else:
    raise AssertionError('Noncoprime periods must be rejected by this constructor')

| Correspondence | N | K | Both matrix identities with shared P1 |
| --- | --- | --- | --- |
| BB144 -> generalized toric | 144 | 12 | True |
| GB period 15 -> BB periods (3,5) | 30 | 0 | True |

Expected refusal for periods (4,6): Coprime periods are needed for a single cycle of length LM


## 5. Optional: distinguish periodic and open boundaries

Boundary checks are part of the construction. A periodic toric example and
an open repetition-chain HGP can both have distance three while having
different N and K. The latter is a small surface code; it is not the paper's
planar88 or tile288/tile512 construction.

In [7]:
rows = []
for name, instance, expected in [
    ('Periodic toric 3x3', toric(3, 3), (18, 2, 3)),
    ('Open repetition-chain HGP', planar_surface(3, 3), (13, 1, 3)),
]:
    result = instance.distance(3)
    assert result['status'] == 'exact'
    assert (instance.n, instance.k, result['value']) == expected
    rows.append([name, instance.n, instance.k, f"D = {result['value']}"])
show_table(["Boundary construction", "N", "K", "Local distance"], rows)

| Boundary construction | N | K | Local distance |
| --- | --- | --- | --- |
| Periodic toric 3x3 | 18 | 2 | D = 3 |
| Open repetition-chain HGP | 13 | 1 | D = 3 |

## 6. What this establishes

The notebook verifies finite bicycle distances, the BB144 dimension by two
methods, direct-sum permutations, and the stated translation/CRT maps.
The exact distance of the reference BB144 remains uncertified here.
Open-boundary instances need their own complete check data; deleting periodic
incidences does not justify retaining the old parameters.

Implementation: [constructors and quotient dimension](../qldpc_generator/constructions.py),
[permutations](../qldpc_generator/equivalences.py), and
[paper-instance tests](../tests/test_paper_cases.py).
Return to [06 — Coverage and validation summary](06-paper-validation.ipynb).